# Configuration

In [4]:
library(here)
library(tidyverse)
library(haven)
library(labelled)
library(arrow)
library(data.table)

source(here("src", "data_wrangling", "data_cleaning.R"))
data_dir <- here("data", "raw", "predimar_miguelgomez.dta")

# Load the data
This data set stores clinical information from patients in the PREDIMAR study. The aim of this project is to build a machine learning model capable of predicting AF recurrence based on medical data.

Let's load the data set:

In [5]:
global <- read_dta(data_dir)

Since it is a STATA file, the labels need to be removed:

In [6]:
original_clinical_data <- global |> remove_labels() 

# Data collection

The data set consists of thousands of features referring over 700 patients. The ones taken into account are sociodemographic, clinical and nutritional characteristics:

- **Sociodemographic data**
    - `sexo`: gender ($0$ male, $1$ female).
    - `edad`: age.
    - `fum_0m`: whether the patient is/was a smoker ($0$ no, $1$ yes).

* **Clinical data**
    - `code`: patient tracking identifier.
    - `nodo`: assigned hospital center.
    - `interv`: assigned cohort ($0$ control, $1$ intervention).
    - `bmi_autoref_0m`: Body Mass Index ($0$ normal weight, $25$ overweight, $30$ obese).
    - `cintura_0m`: waist circumference ($cm$).
    - `altura_0m:`height ($cm$).
    - `mettotal_0m`: Met (physical activity measurement).
    - `glu_new0m`: glucose level ($mg/dL$).
    - `diabetest1_0m`:type 1 diabetes ($0$ no, $1$ yes).
    - `diabetest2_0m`:type 2 diabetes ($0$ no, $1$ yes).
    - `hdl_new0m`: HDL cholesterol level ($mg/dL$).
    - `trig_new0m`: triglycerides level ($mg/dL$).
    - `hipercol_0m`: whether the patient suffers from dyslipidemia ($0$ no, $1$ yes).
    - `hipolipemiantes_0m`: whether the patient takes hypolipidemic medication ($0$ no, $1$ yes).
    - `saos_0m`: whether the patient suffers from sleep apnea ($0$ no, $1$ yes). 
    - `insurenal_0m`: whether the patient suffers from renal insufficiency ($0$ no, $1$ yes).
    - `hta_0m`: hypertension ($0$ no, $1$ yes).
    - `epoc_0m`: whether the patient suffers from chronic obstructive pulmonary disease ($0$ no, $1$ yes).
    - `ictus_0m`: whether the patient has suffered from a stroke ($0$ no, $1$ yes).
    - `ic_v00`: whether the patient has suffered from heart failure ($0$ no, $1$ yes).
    - `miocardiopat_0m_imp`: whether the patient suffers from cardiomyopathy ($0$ no, $1$ yes).
    - `faa_0m_new`: whether the patient is having antiarrhythmic drugs ($0$ no, $1$ yes).
    - `dilat_tot4_imp`: left atrial enlargement ($0$ normal, $1$ mild, $2$ moderate, $3$ severe).
    - `eco_diamai_0m`: left atrial diameter ($cm$).
    - `eco_areaai_0m`: left atrial area ($cm^2)$.
    - `eco_volumenai_0m`: left atrial volume ($mL)$.
    - `eco_fe_0m`: left atrial ejection fraction ($\%$).
    - `tipofa`: type of Atrial Fibrillation ($0$ paroxysmal, $1$ persistent).
    - `tiempo_fa_ablac`: years since the first AF diagnosis.
    - `ablacion_previa`: whether the patient has undergone catheter ablation previously to this trial ($0$ no, $1$ yes).
    - `event_blank`: whether there has been AF recurrence during the blanking period ($0$ no, $1$ yes).
    - `event18m_conkardia`: whether there has been AF recurrence between $3$ and $18$ months since the catheter ablation ($0$ no, $1$ yes).

In [7]:
clinical_data_selected <- original_clinical_data |> 
    select(
        #=======================================================================
        # SOCIODEMOGRAPHIC DATA
        #=======================================================================
        sexo,
        edad,
        fum_0m,
        
        #=======================================================================
        # CLINICAL DATA
        #=======================================================================
        # Tracking & Trial Context
        code,
        nodo,
        interv,
        
        # Lifestyle & General Comorbidities
        bmi_autoref_0m,
        cintura_0m,
        altura_0m,
        mettotal_0m,
        glu_new0m,
        diabetest1_0m,
        diabetest2_0m,
        hdl_new0m,
        trig_new0m,
        hipercol_0m,
        hipolipemiantes_0m,
        saos_0m,
        insurenal_0m,
        hta_0m,
        epoc_0m,
        ictus_0m,
        
        # Specific Cardiac Conditions & Echo Structural Measurements
        ic_v00,
        miocardiopat_0m_imp,
        faa_0m_new,
        dilat_tot4_imp,
        eco_diamai_0m,
        eco_areaai_0m,
        eco_volumenai_0m,
        eco_fe_0m,
        
        # Atrial Fibrillation History & Dynamics
        tipofa,
        tiempo_fa_ablac,
        ablacion_previa,
        event_blank,
        
        # Target class
        event18m_conkardia
    )

head(clinical_data_selected)

sexo,edad,fum_0m,code,nodo,interv,bmi_autoref_0m,cintura_0m,altura_0m,mettotal_0m,⋯,dilat_tot4_imp,eco_diamai_0m,eco_areaai_0m,eco_volumenai_0m,eco_fe_0m,tipofa,tiempo_fa_ablac,ablacion_previa,event_blank,event18m_conkardia
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,75.06,0,1013,1,1,37.57763,NA,159,4.841667,⋯,3,5.0,NA,35.0,0,1,4.3860369,0,0,0
0,28.01,0,1015,1,0,25.28011,92,179,36.714169,⋯,1,3.5,NA,56.0,0,1,0.3778234,0,0,0
1,70.10,0,1018,1,0,25.81663,94,177,11.582867,⋯,1,4.1,27.5,61.0,0,0,0.3066393,0,0,0
0,73.40,2,1021,1,0,24.85795,102,176,39.271667,⋯,0,25.0,NA,24.7,0,0,0.4763860,0,0,1
0,66.01,0,1029,1,1,29.86055,NA,185,17.056667,⋯,3,42.0,NA,74.1,0,0,9.1745377,1,0,0
0,72.98,0,1030,1,0,23.98687,NA,180,23.619167,⋯,1,4.4,NA,NA,0,0,2.6146474,1,0,0


# Data cleaning
Lets label the categorical data using factors:

In [8]:
clinical_data_modified <- clinical_data_selected |> 
    mutate(
        # ======================================================================
        # SOCIODEMOGRAPHIC DATA
        # ======================================================================
        sexo = factor(
            sexo, 
            levels = c(0, 1), 
            labels = c("male", "female"),
            ordered = FALSE
        ),
        
        fum_0m = factor(
            fum_0m,
            levels = c(0, 2, 1),
            labels = c("never", "current", "former"),
            ordered = TRUE
        ),
        
        # ======================================================================
        # CLINICAL DATA
        # ======================================================================
        # Tracking & Trial Context
        nodo = factor(
            nodo,
            levels = c(1, 2, 3, 4),
            labels = c("Pamplona", "Madrid", "Granada", "Alicante"),
            ordered = FALSE
        ),
        
        interv = factor(
            interv,
            levels = c(0, 1),
            labels = c("control", "intervention"),
            ordered  = FALSE
        ),

        # Lifestyle & General Comorbidities
        diabetest1_0m = factor(
            diabetest1_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        diabetest2_0m = factor(
            diabetest2_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        hipercol_0m = factor(
            hipercol_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        hipolipemiantes_0m = factor(
            hipolipemiantes_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        saos_0m = factor(
            saos_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        insurenal_0m = factor(
            insurenal_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        hta_0m = factor(
            hta_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        epoc_0m = factor(
            epoc_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        ictus_0m = factor(
            ictus_0m,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        # Specific Cardiac Conditions & Echo Measurements
        ic_v00 = factor(
            ic_v00,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        miocardiopat_0m_imp = factor(
            miocardiopat_0m_imp,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        faa_0m_new = factor(
            faa_0m_new,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        dilat_tot4_imp = factor(
            dilat_tot4_imp,
            levels = c(0, 1, 2, 3),
            labels = c("normal", "mild", "moderate", "severe"),
            ordered = TRUE
        ),

        eco_fe_0m = factor(
            eco_fe_0m,
            levels = c(0, 1, 2, 3),
            labels = c("normal", "slightly abnormal",
                       "moderately abnormal", "severely abnormal"),
            ordered = TRUE
        ),
        
        # =====================================================================
        # CLINICAL DATA: Atrial Fibrillation History & Dynamics
        # =====================================================================
        tipofa = factor(
            tipofa,
            levels = c(0, 1),
            labels = c("persistent", "paroxysmal"),
            ordered = FALSE
        ),
        
        ablacion_previa = factor(
            ablacion_previa,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        event_blank = factor(
            event_blank,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        ),
        
        # TARGET CLASS
        event18m_conkardia = factor(
            event18m_conkardia,
            levels = c(0, 1),
            labels = c("no", "yes"),
            ordered = FALSE
        )
    ) #|> 
    
    # # Reorder the factor levels appropriately
    # mutate(
    #     tipofa = fct_relevel(tipofa, "paroxysmal")
    # )

head(clinical_data_modified)

sexo,edad,fum_0m,code,nodo,interv,bmi_autoref_0m,cintura_0m,altura_0m,mettotal_0m,⋯,dilat_tot4_imp,eco_diamai_0m,eco_areaai_0m,eco_volumenai_0m,eco_fe_0m,tipofa,tiempo_fa_ablac,ablacion_previa,event_blank,event18m_conkardia
<fct>,<dbl>,<ord>,<dbl>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<ord>,<dbl>,<dbl>,<dbl>,<ord>,<fct>,<dbl>,<fct>,<fct>,<fct>
female,75.06,never,1013,Pamplona,intervention,37.57763,NA,159,4.841667,⋯,severe,5.0,NA,35.0,normal,paroxysmal,4.3860369,no,no,no
male,28.01,never,1015,Pamplona,control,25.28011,92,179,36.714169,⋯,mild,3.5,NA,56.0,normal,paroxysmal,0.3778234,no,no,no
female,70.10,never,1018,Pamplona,control,25.81663,94,177,11.582867,⋯,mild,4.1,27.5,61.0,normal,persistent,0.3066393,no,no,no
male,73.40,current,1021,Pamplona,control,24.85795,102,176,39.271667,⋯,normal,25.0,NA,24.7,normal,persistent,0.4763860,no,no,yes
male,66.01,never,1029,Pamplona,intervention,29.86055,NA,185,17.056667,⋯,severe,42.0,NA,74.1,normal,persistent,9.1745377,yes,no,no
male,72.98,never,1030,Pamplona,control,23.98687,NA,180,23.619167,⋯,mild,4.4,NA,NA,normal,persistent,2.6146474,yes,no,no


Display the current names of the columns:

In [9]:
names(clinical_data_modified)

[1] "sexo"                "edad"                "fum_0m"             
 [4] "code"                "nodo"                "interv"             
 [7] "bmi_autoref_0m"      "cintura_0m"          "altura_0m"          
[10] "mettotal_0m"         "glu_new0m"           "diabetest1_0m"      
[13] "diabetest2_0m"       "hdl_new0m"           "trig_new0m"         
[16] "hipercol_0m"         "hipolipemiantes_0m"  "saos_0m"            
[19] "insurenal_0m"        "hta_0m"              "epoc_0m"            
[22] "ictus_0m"            "ic_v00"              "miocardiopat_0m_imp"
[25] "faa_0m_new"          "dilat_tot4_imp"      "eco_diamai_0m"      
[28] "eco_areaai_0m"       "eco_volumenai_0m"    "eco_fe_0m"          
[31] "tipofa"              "tiempo_fa_ablac"     "ablacion_previa"    
[34] "event_blank"         "event18m_conkardia"

Let's modify them:

In [10]:
new_names <- c(
    "sex", "age", "smoking_status", "code", "center", "intervention", "BMI", 
    "waist_circumference", "height", "Met", "glucose", "type1_diabetes", 
    "type2_diabetes", "HDL", "triglicerides", "hypercholesterolemia", 
    "hypolipidemic_meds", "OSA", "renal_insuf", "hypertension", "COPD", 
    "stroke", "heart_failure", "cardiomyopathy", "antirrythmic_meds", 
    "LA_enlargment", "LAD", "LAA", "LAV", "LVEF", "AF_type", 
    "AF_ablation_time", "previous_ablation", "ERAF", 
    "AF_recurrence"
    )

clinical_data_processed <- clinical_data_modified |> setnames(
    old = names(clinical_data_modified), 
    new = new_names
    )

# Feature engineering
As an alternative to the `BMI` feature, we can compute the waist-to-height ratio to assess obesity:

In [11]:
clinical_data_processed <- clinical_data_processed |> 
    mutate(
        waist_height_ratio = waist_circumference / height,
        .after = BMI
    ) |> 
    select(-c(waist_circumference, height))

We can also compute the HATCH score for each patient:

In [12]:
clinical_data_processed <- compute_hatch_score(clinical_data_processed)

# Save data
Save the whole data set:

In [13]:
whole_data_output_dir <- here("data", "intermediate", "whole_clinical_data.parquet")

write_parquet(
    clinical_data_processed, 
    whole_data_output_dir
    )

head(clinical_data_processed)

sex,age,smoking_status,code,center,intervention,BMI,waist_height_ratio,Met,glucose,⋯,LAD,LAA,LAV,LVEF,AF_type,AF_ablation_time,previous_ablation,ERAF,AF_recurrence,hatch_score
<fct>,<dbl>,<ord>,<dbl>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<ord>,<fct>,<dbl>,<fct>,<fct>,<fct>,<dbl>
female,75.06,never,1013,Pamplona,intervention,37.57763,NA,4.841667,114.59,⋯,5.0,NA,35.0,normal,paroxysmal,4.3860369,no,no,no,2
male,28.01,never,1015,Pamplona,control,25.28011,0.5139665,36.714169,100.12,⋯,3.5,NA,56.0,normal,paroxysmal,0.3778234,no,no,no,0
female,70.10,never,1018,Pamplona,control,25.81663,0.5310734,11.582867,102.81,⋯,4.1,27.5,61.0,normal,persistent,0.3066393,no,no,no,0
male,73.40,current,1021,Pamplona,control,24.85795,0.5795455,39.271667,97.67,⋯,25.0,NA,24.7,normal,persistent,0.4763860,no,no,yes,0
male,66.01,never,1029,Pamplona,intervention,29.86055,NA,17.056667,82.60,⋯,42.0,NA,74.1,normal,persistent,9.1745377,yes,no,no,2
male,72.98,never,1030,Pamplona,control,23.98687,NA,23.619167,111.29,⋯,4.4,NA,NA,normal,persistent,2.6146474,yes,no,no,0


Save a subset with only the features used for the HATCH score:

In [14]:
hatch_data_output_dir <- here("data", "clean", "hatch_data.parquet")

hatch_data <- clinical_data_processed |> 
    select(
        code, age, hypertension, COPD, stroke, heart_failure, AF_type, hatch_score, AF_recurrence
        )
write_parquet(
    hatch_data, 
    hatch_data_output_dir
    )

head(hatch_data)

code,age,hypertension,COPD,stroke,heart_failure,AF_type,hatch_score,AF_recurrence
<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<fct>
1013,75.06,yes,no,no,no,paroxysmal,2,no
1015,28.01,no,no,no,no,paroxysmal,0,no
1018,70.10,no,no,no,no,persistent,0,no
1021,73.40,no,no,no,no,persistent,0,yes
1029,66.01,yes,no,no,no,persistent,2,no
1030,72.98,no,no,no,no,persistent,0,no
